# **Code Search Agent** with LangChain and Gemini (GitHub Toolkit)

**Install packages**

In [7]:
!pip install langchain -qU
!pip install langchain-google-genai -qU
!pip install langchain-community -qU
!pip install pygithub -qU
!pip install langchainhub -qU
!pip install tiktoken -qU

In [9]:
import os
from google.colab import userdata

**Upload the private key file**

In [10]:
from google.colab import files

# Upload the .pem file you downloaded from your GitHub App settings
uploaded = files.upload()
pem_filename = list(uploaded.keys())[0]
print(f"Uploaded: {pem_filename}")

Saving code-search-agent.2026-07-11.private-key.pem to code-search-agent.2026-07-11.private-key.pem
Uploaded: code-search-agent.2026-07-11.private-key.pem


In [12]:
# Gemini API key
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# GitHub Personal Access Token — get one at https://github.com/settings/tokens
# (classic token, no special scopes needed for public repo read access)
os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = userdata.get('GITHUB_PAT')

# GitHub App ID — found at the top of your app's settings page
os.environ["GITHUB_APP_ID"] = userdata.get('GITHUB_APP_ID')

# Path to the private key file you just uploaded
os.environ["GITHUB_APP_PRIVATE_KEY"] = f"/content/{pem_filename}"

# The public repo you want the agent to search (owner/repo format)
os.environ["GITHUB_REPOSITORY"] = "kavishannip/Ai-Diagrams"

 **Initialize Gemini LLM**

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

**Initialize GitHub Toolkit**

In [21]:
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

github = GitHubAPIWrapper()
toolkit = GitHubToolkit.from_github_api_wrapper(github)

allowed_tools = {"Search code", "Read File"}
tools = [t for t in toolkit.get_tools() if t.name in allowed_tools]

# Gemini requires tool names with no spaces — rename them
for t in tools:
    t.name = t.name.replace(" ", "_").lower()

for tool in tools:
    print(tool.name)

read_file
search_code


**Load Prompt Template**

In [22]:
system_prompt = """You are a helpful coding assistant with access to GitHub tools.
Use the tools to search code and read files in the repository to answer
the user's question accurately. Only reference files and content you
actually retrieved from the tools — never guess file paths."""

**Create Agent**

In [23]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

**Invoke the agent**

In [24]:
response = agent.invoke({"messages": [("user", "Search the repo for where 'DiagramResult' is defined and tell me the file path.")]})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The `DiagramResult` component is defined in the file: `src/components/DiagramResult.jsx`.', 'extras': {'signature': 'EjQKMgERTTIP8kBPP78geZW0EhqMGrSvRzrEYTFuukeioWu4PyZ+BBwF2VcA7x3TQsK8htcn'}}]


In [26]:
response = agent.invoke({"messages": [("user", "Search the repo for where 'DiagramResult' is defined and tell me the file path.")]})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Search the repo for where 'DiagramResult' is defined and tell me the file path.
================================== Ai Message ==================================

[]
Tool Calls:
  search_code (wP4OzcFt)
 Call ID: wP4OzcFt
  Args:
    search_query: class DiagramResult
================================= Tool Message =================================
Name: search_code

Showing top 5 of 5 results:
Filepath: `src/components/DiagramResult.jsx`
File contents: "use client";
import React, { useState, useEffect, useRef } from "react";
import {
  ArrowsPointingOutIcon,
  ArrowsPointingInIcon,
  ArrowDownTrayIcon,
} from "@heroicons/react/24/outline";
import DiagramLoading from "./Loaders/DiagramLoading";
import { RiTable2 } from "react-icons/ri";
import { HiArrowsUpDown, HiArrowsRightLeft } from "react-icons/hi2";
import { PiLayoutDuotone } from "react-icons/pi";
import { UserButton } from "@clerk/nextjs";
import { us

In [27]:
response = agent.invoke({"messages": [("user", "How user authentication works in this repo, explain the process")]})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The user authentication in this repository is handled by **Clerk**, a popular third-party authentication service.\n\n### How it works:\n\n1.  **Integration**: The application uses the `@clerk/nextjs` SDK to manage user sessions, sign-in, and sign-up flows.\n2.  **Configuration**: Authentication is configured via environment variables in the `.env.local` file:\n    *   `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY`\n    *   `CLERK_SECRET_KEY`\n    *   Various URLs for redirecting users after sign-in/sign-up.\n3.  **UI Components**: The application uses Clerk\'s pre-built UI components to handle the user experience. For example, in `src/components/DiagramResult.jsx`, the `<UserButton />` component is used to provide a user profile interface:\n    ```javascript\n    import { UserButton } from "@clerk/nextjs";\n    // ...\n    <UserButton userProfileUrl="/profile" />\n    ```\n4.  **User Management**: Clerk manages the user database, session tokens, and authentication state

In [28]:
response = agent.invoke({"messages": [("user", "How Diagram generation process is happened, from user prompt to generate diagram")]})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The diagram generation process in this application follows a structured flow from user input to the final rendered image. Based on the codebase, here is the step-by-step breakdown:\n\n### 1. User Input\nThe process begins when a user provides input through one of two primary methods:\n*   **Text Prompt:** The user describes a diagram in natural language (e.g., "Create a sequence diagram for user login").\n*   **Image Upload:** The user uploads an image file that contains a diagram.\n\n### 2. AI Processing (Generation)\nThe application uses Google\'s Gemini AI models to interpret the user\'s input and generate the necessary code to create the diagram.\n*   **Text-to-PlantUML:** The text prompt is sent to an API endpoint (e.g., `/api/text-to-plantuml`) where the Gemini model (typically Gemini 2.5 Flash for speed) analyzes the request and generates the corresponding **PlantUML code**.\n*   **Image-to-PlantUML:** The uploaded image is sent to an API endpoint (e.g